In [1]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

In [2]:
ner = pipeline(
    "ner",
    model="samrawal/bert-base-uncased_clinical-ner",
    aggregation_strategy="first",
    device="mps"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
merged = pd.read_parquet("../data/processed/merged_sepsis_notes.parquet")

In [4]:
text = merged["brief hospital course"].iloc[0]
entities = ner(text)
entities = [r for r in entities if r["score"] > 0.85]
for r in entities:
    print(r["entity_group"], r["word"], round(r["score"], 3))

problem htn 0.99
problem hld 0.947
problem intermittent chest pain 0.997
problem chemical stress abnormal 0.996
test lhc 0.95
problem 99 % lm lesion 0.997
treatment iabp 0.981
treatment further management 0.981
treatment cabg 0.911
treatment cabg 0.985
treatment the procedure 0.991
test invasive monitoring 0.981
treatment iabp 0.981
treatment beta blocker 0.998
treatment chest tubes 0.997
treatment pacing wires 0.989
problem complication 0.973
problem the wound 0.996
problem pain 0.995
treatment oral analgesics 0.997


In [5]:
def extract_entities_from_row(row, sections, ner, threshold=0.85):
    records = []
    for section in sections:
        text = row.get(section, "")
        if not isinstance(text, str) or len(text.strip()) == 0:
            continue
        try:
            entities = ner(text[:512])  # truncate for now
            for r in entities:
                if r["score"] >= threshold:
                    records.append({
                        "subject_id": row["subject_id"],
                        "hadm_id": row["hadm_id"],
                        "section": section,
                        "entity": r["word"],
                        "label": r["entity_group"],
                        "score": round(r["score"], 3),
                        "race": row["race"],
                        "insurance": row["insurance"],
                        "gender": row["gender"],
                        "anchor_age": row["anchor_age"],
                        "los": row["los"],
                        "expired": row["hospital_expire_flag"]
                    })
        except Exception as e:
            print(f"Error on hadm_id {row['hadm_id']}: {e}")
            continue
    return records

In [6]:
sections_of_interest = [
    "brief hospital course",
    "medications on admission",
    "discharge medications",
    "discharge diagnosis",
]

records = []
for _, row in tqdm(merged.iterrows(), total=len(merged)):
    records.extend(extract_entities_from_row(row, sections_of_interest, ner))

100%|██████████| 32513/32513 [1:41:51<00:00,  5.32it/s]  


In [7]:
entities_df = pd.DataFrame(records)
entities_df.to_parquet("../data/processed/entities.parquet", index=False)